# Pandas and NumPy Interview Preparation

This notebook contains runnable examples using small synthetic datasets. It focuses on interview concepts: array operations, selection, data cleaning, grouping, merging, reshaping, dates, and performance.

Run cells in order, inspect the outputs, then change the data to test edge cases.

In [ ]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 1. NumPy arrays, shape, dtype, and axis

In [ ]:
arr = np.arange(12).reshape(3, 4)
print("Array:\n", arr)
print("shape:", arr.shape, "| ndim:", arr.ndim, "| dtype:", arr.dtype)
print("column means:", arr.mean(axis=0))
print("row sums:", arr.sum(axis=1))


## 2. Indexing, masks, vectorization, and broadcasting

In [ ]:
print("second row, last two columns:", arr[1, -2:])
print("values greater than 7:", arr[arr > 7])

a = np.array([[1], [2], [3]])       # shape (3, 1)
b = np.array([10, 20, 30, 40])      # shape (4,)
print("broadcast result shape:", (a + b).shape)
print(a + b)


In [ ]:
values = np.array([10.0, 20.0, 30.0])
span = values.max() - values.min()
scaled = np.zeros_like(values) if span == 0 else (values - values.min()) / span
print("min-max scaled:", scaled)


## 3. Create a sample DataFrame

In [ ]:
employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5, 6, 7],
    "name": ["Asha", "Ravi", "Mina", "Omar", "Leah", "Arun", "Isha"],
    "department": ["Data", "AI", "Data", "Sales", "AI", None, "Sales"],
    "score": [91, 82, 91, 74, np.nan, 88, 69],
    "hire_date": ["2022-01-10", "2021-06-03", "2023-02-14", "2020-08-25",
                  "2024-03-02", "2022-11-30", "2024-07-18"]
})
employees


## 4. Selection and Boolean filtering

In [ ]:
employees.loc[
    (employees["score"] >= 80) & (employees["department"].isin(["Data", "AI"])),
    ["name", "department", "score"]
]


In [ ]:
# Label-based and position-based selection
print("loc label example:\n", employees.loc[0:2, ["name", "score"]])
print("iloc position example:\n", employees.iloc[:3, :3])


## 5. Missing values, duplicate rows, and string cleaning

In [ ]:
print("Missing values by column:\n", employees.isna().sum())
median_score = employees["score"].median()
employees_clean = employees.copy()
employees_clean["score"] = employees_clean["score"].fillna(median_score)
employees_clean["department_clean"] = (
    employees_clean["department"].astype("string").str.strip().str.lower()
)
employees_clean.head()


## 6. Grouping: agg versus transform

In [ ]:
summary = employees.groupby("department", dropna=False).agg(
    mean_score=("score", "mean"),
    row_count=("score", "size"),
    scored_count=("score", "count")
)
summary


In [ ]:
employees_with_mean = employees.copy()
employees_with_mean["department_mean"] = (
    employees_with_mean.groupby("department")["score"].transform("mean")
)
employees_with_mean[["name", "department", "score", "department_mean"]]


## 7. Merge and validate join cardinality

In [ ]:
departments = pd.DataFrame({
    "department": ["Data", "AI", "Sales", "Legal"],
    "manager": ["Nisha", "Karan", "Meera", "Dev"]
})
merged = employees.merge(
    departments, on="department", how="left",
    validate="many_to_one", indicator=True
)
merged[["name", "department", "manager", "_merge"]]


## 8. Reshape: pivot_table and melt

In [ ]:
sales = pd.DataFrame({
    "date": pd.to_datetime(["2025-01-01", "2025-01-01", "2025-02-01", "2025-02-01"]),
    "product": ["Book", "Pen", "Book", "Pen"],
    "sales": [100, 30, 120, 45]
})
wide = sales.pivot_table(index="date", columns="product", values="sales", aggfunc="sum", fill_value=0)
print("Wide format:\n", wide)
print("Long format:\n", wide.reset_index().melt(id_vars="date", var_name="product", value_name="sales"))


## 9. Datetime features and rolling windows

In [ ]:
dated = employees.copy()
dated["hire_date"] = pd.to_datetime(dated["hire_date"], errors="coerce")
dated["hire_year"] = dated["hire_date"].dt.year
dated["hire_month"] = dated["hire_date"].dt.month
dated[["name", "hire_date", "hire_year", "hire_month"]]


In [ ]:
daily = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=6, freq="D"),
    "revenue": [100, 120, 90, 140, 160, 130]
})
daily["rolling_mean_3"] = daily["revenue"].rolling(window=3, min_periods=1).mean()
daily


## 10. Memory and merge diagnostics

In [ ]:
print("Memory usage (bytes):")
print(employees.memory_usage(deep=True))
print("\nDuplicate employee IDs:", employees["employee_id"].duplicated().sum())
print("\nMerge row count:", len(merged), "| original employee row count:", len(employees))


## Mini challenges

1. Return all rows tied for the highest score within each department.
2. Add a Boolean column that flags scores of 80 or above using `.loc`.
3. Identify employee rows without a matching department manager.
4. Change one department key to be duplicated and observe why a validated merge can fail.
5. Create a vectorized NumPy expression that replaces negative values with zero.

Try the tasks in `practice.md` and use `solution.md` only after attempting them.